# Sudoku-12-Z3-Rust : Z3 SMT Solver (Rust)

**Navigation** : [<< OR-Tools Python](Sudoku-10-ORTools-Python.ipynb) | [Index](README.md) | [Infer Python >>](Sudoku-15-Infer-Python.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Modéliser le Sudoku avec un solveur SMT (Z3) depuis Rust
2. Comprendre la différence entre SAT et SMT
3. Utiliser des BitVectors pour optimiser les représentations
4. Comparer Z3 Int avec Z3 BitVector

**Durée estimée** : ~10 min | **Prérequis** : [Sudoku-00 Environment](Sudoku-00-Environment-CSharp.ipynb)

***

Ce notebook implémente un solveur de Sudoku utilisant **Z3** (prouveur SMT issu de Microsoft Research, désormais open-source sous licence MIT), un solveur **SMT** (Satisfiable Modulo Theories), écrit en **Rust**.

C'est le jumeau Rust du carnet `Sudoku-12-Z3-Python.ipynb` : même problème, même encodage (81 variables, 27 contraintes `Distinct`, même corpus de puzzles), même moteur de décision — seule la langue et l'API changent.

## Installation

En Python, on installe le paquet `z3-solver`. En Rust, la dépendance se déclare dans la première cellule du carnet, une seule fois :

```rust
:dep z3 = "0.24.0"
```

La crate `z3` 0.24.0 ne déclare **aucune feature par défaut** : sans option, la dépendance `z3-sys` lie une `libz3` **système** (mesure sur ce poste : Z3 5.3.3). La feature `gh-release` -- celle retenue par les carnets Rust Z3 déjà présents dans le dépôt (`CSP-10-Optimisation-Z3-Rust.ipynb`) -- télécharge une release précompilée de Z3 et rendrait le carnet autonome, mais l'asset Windows de cette release est **MSVC-only** : sur un poste dont le toolchain rust par défaut est `x86_64-pc-windows-gnu`, l'édition de liens échoue (`could not find native static library z3`), alors qu'elle réussit avec `stable-x86_64-pc-windows-msvc` (mesure : édition complète en 19,56 s). Le carnet est donc livré dans sa forme liée au Z3 système ; l'alignement de toolchain qui permettrait `gh-release` fait l'objet d'une issue de suivi.

La directive `:dep` appartient au noyau `evcxr` : elle ajoute la crate au projet temporaire du noyau et la compile. Elle se place **en tête de carnet** — un `:dep` plus bas recompilerait tout et effacerait les variables typées des cellules précédentes.


In [2]:
:dep z3 = "0.24.0"

use z3::ast::{Ast, Bool, BV, Int};
use z3::{Optimize, SatResult, Solver};

// La dépendance z3 est déclarée UNE SEULE FOIS, en tête de carnet :
// un :dep plus bas recompilerait tout et effacerait les variables typées.
println!("Z3 importe avec succes");


Z3 importe avec succes


## 1. Solveur Z3 SMT

**Z3** est un solveur **SMT** (*Satisfiability Modulo Theories*) développé à l'origine par **Microsoft Research** (Leonardo de Moura et Nikolaj Bjørner), désormais publié en **open-source sous licence MIT** dans le dépôt [`Z3Prover/z3`](https://github.com/Z3Prover/z3). Il combine la puissance des solveurs SAT avec des théories mathématiques (arithmétique, tableaux, bitvectors, etc.).

En Rust, on ne parle pas à `z3` par le binding `pyz3` mais par la **crate** `z3`, qui lie la même bibliothèque C. L'API est la transposée directe de l'API Python :

| Python (`z3-solver`) | Rust (crate `z3`) |
|---|---|
| `Int('x_0_0')` | `Int::new_const("x_0_0")` |
| `solver.add(c)` | `solver.assert(&c)` |
| `solver.check() == sat` | `solver.check() == SatResult::Sat` |
| `solver.model()` | `solver.get_model()` (`Option<Model>`) |
| `model.evaluate(x)` | `model.eval(&x, true)` (`Option<Int>`) |
| `Distinct([...])` | `Int::distinct(&[...])` |
| `x >= 1` | `x.ge(&Int::from_i64(1))` |


> **Source et licence.** Z3 est un prouvent de théorèmes issu de Microsoft Research, désormais libre (licence **MIT**) sur [`github.com/Z3Prover/z3`](https://github.com/Z3Prover/z3). Référence canonique : Leonardo de Moura et Nikolaj Bjørner, *Z3: An Efficient SMT Solver*, in *Tools and Algorithms for the Construction and Analysis of Systems (TACAS)*, 2008. Ce notebook distille cette source au travers du prisme pédagogique du Sudoku : il ne reproduit ni l'article ni la documentation officielle, mais démontre l'API de la crate `z3` (`Int`, `BitVec`, `Solver`, `Optimize`) sur un exemple concret.


### Différence entre SAT et SMT

| SAT | SMT |
|-----|-----|
| Variables booléennes uniquement | Variables typées (Int, Real, BitVec, Array...) |
| Clauses propositionnelles | Formules de premier ordre |
| `(x OR y) AND (NOT x OR z)` | `x + y > 10 AND x < 5` |


## 2. Configuration du chemin vers les puzzles


In [3]:
use std::path::PathBuf;

// Localiser le dossier Puzzles : le chemin absolu dépend de la machine,
// on n'affiche donc que le nom du dossier (sortie portable). Le chemin
// complet reste disponible dans `puzzles_dir` pour les entrées/sorties.
let puzzles_dir: PathBuf = PathBuf::from("Puzzles");

// Vérifier que le dossier existe
if puzzles_dir.exists() {
    let nom = puzzles_dir
        .file_name()
        .map(|n| n.to_string_lossy().to_string())
        .unwrap_or_default();
    println!("Dossier Puzzles: {}/", nom);

    let mut noms: Vec<String> = Vec::new();
    match std::fs::read_dir(&puzzles_dir) {
        Ok(entrees) => {
            for entree in entrees.flatten() {
                let chemin = entree.path();
                if chemin.extension().map(|e| e.to_string_lossy() == "txt").unwrap_or(false) {
                    if let Some(fichier) = chemin.file_name() {
                        noms.push(fichier.to_string_lossy().to_string());
                    }
                }
            }
        }
        Err(e) => println!("Lecture du dossier impossible : {}", e),
    }
    noms.sort();

    // Format de liste Python (apostrophes simples) : la ligne reste
    // comparable à celle du jumeau Python.
    let liste: Vec<String> = noms.iter().map(|n| format!("'{}'", n)).collect();
    println!("Fichiers disponibles: [{}]", liste.join(", "));
} else {
    println!("ATTENTION: Dossier Puzzles non trouve");
};


Dossier Puzzles: Puzzles/


Fichiers disponibles: ['Sudoku_Easy51.txt', 'Sudoku_gen_difficile.txt', 'Sudoku_gen_facile.txt', 'Sudoku_gen_moyen.txt', 'Sudoku_hardest.txt', 'Sudoku_top95.txt']


## 3. Solveurs Z3 : Int et BitVector

Nous présentons deux approches pour encoder le Sudoku avec Z3 :

- **Z3 Int** : variables entières symboliques (`Int`), encodage classique.
- **Z3 BitVec** : variables BitVec 4 bits, optimisation pour plages bornées.

Nous définissons d'abord une représentation de la grille, puis comparons les deux encodages sur les mêmes puzzles.


Représentation de la grille Sudoku avec fonctions de chargement.


In [4]:
use std::path::Path;

{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }
    /// Lecture des puzzles : l'erreur d'ouverture est TRAITÉE (`Result`),
    /// jamais avalée.
    fn charger_puzzles(chemin: &std::path::Path, max_puzzles: Option<usize>) -> Result<Vec<String>, std::io::Error> {
        let contenu = std::fs::read_to_string(chemin)?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect::<String>());
                if let Some(max) = max_puzzles {
                    if puzzles.len() >= max {
                        break;
                    }
                }
            }
        }
        Ok(puzzles)
    }

    // Charger les puzzles
    let dir = Path::new("Puzzles");
    let easy_puzzles = match charger_puzzles(&dir.join("Sudoku_Easy51.txt"), Some(10)) {
        Ok(v) => v,
        Err(e) => {
            println!("Lecture des puzzles faciles impossible : {}", e);
            Vec::new()
        }
    };
    let hard_puzzles = match charger_puzzles(&dir.join("Sudoku_hardest.txt"), None) {
        Ok(v) => v,
        Err(e) => {
            println!("Lecture des puzzles difficiles impossible : {}", e);
            Vec::new()
        }
    };
    println!(
        "Puzzles chargés: {} faciles, {} difficiles",
        easy_puzzles.len(),
        hard_puzzles.len()
    );
};


Puzzles chargés: 10 faciles, 11 difficiles


### Lecture du corpus : trois fichiers, 10 faciles + 11 difficiles

La sortie énumère le contenu du dossier commun de la série — **Easy51, hardest, top95** — puis la classe de chargement en tire **10 faciles et 11 difficiles**. Ce sont les mêmes corpus que le benchmark du carnet de propagation de Norvig (20 faciles à 3,30 ms, 11 difficiles à 4,45 ms par propagation) : les encodages diffèrent, les terrains de jeu non. Le puzzle difficile affiché en tête de test (8 5 . | . . 2 | 4 . . ...) est l'exemplaire sur lequel le solveur `Int` de la section suivante est mesuré : la durée est **remesurée à chaque exécution** par la cellule de code, elle n'est figée ni ici ni dans le carnet jumeau.


Définition du solveur Z3 pour Sudoku.


In [5]:
use std::path::Path;
use std::time::Instant;
use z3::ast::{Ast, Int};
use z3::{SatResult, Solver};

{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }
    /// Lecture des puzzles : l'erreur d'ouverture est TRAITÉE (`Result`),
    /// jamais avalée.
    fn charger_puzzles(chemin: &std::path::Path, max_puzzles: Option<usize>) -> Result<Vec<String>, std::io::Error> {
        let contenu = std::fs::read_to_string(chemin)?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect::<String>());
                if let Some(max) = max_puzzles {
                    if puzzles.len() >= max {
                        break;
                    }
                }
            }
        }
        Ok(puzzles)
    }

    /// Solveur Sudoku utilisant Z3 SMT.
    struct Z3Solver;

    impl Z3Solver {
        /// Résout le Sudoku avec Z3. La grille est mise à jour en place.
        fn solve(&self, grid: &mut SudokuGrid) -> bool {
            // Créer les variables : X[i][j] est un entier 1-9
            let mut x: Vec<Vec<Int>> = Vec::new();
            for i in 0..9 {
                let mut ligne: Vec<Int> = Vec::new();
                for j in 0..9 {
                    ligne.push(Int::new_const(format!("x_{i}_{j}")));
                }
                x.push(ligne);
            }

            let mut solver = Solver::new();

            // Contraintes de domaine : 1 <= X[i][j] <= 9
            for i in 0..9 {
                for j in 0..9 {
                    solver.assert(&x[i][j].ge(&Int::from_i64(1)));
                    solver.assert(&x[i][j].le(&Int::from_i64(9)));
                }
            }

            // Valeurs initiales
            for i in 0..9 {
                for j in 0..9 {
                    if grid.cells[i][j] != 0 {
                        solver.assert(&x[i][j]._eq(&Int::from_i64(grid.cells[i][j] as i64)));
                    }
                }
            }

            // Contraintes : toutes différentes par ligne
            for i in 0..9 {
                let vars: Vec<Int> = (0..9).map(|j| x[i][j].clone()).collect();
                solver.assert(&Int::distinct(&vars));
            }

            // Contraintes : toutes différentes par colonne
            for j in 0..9 {
                let vars: Vec<Int> = (0..9).map(|i| x[i][j].clone()).collect();
                solver.assert(&Int::distinct(&vars));
            }

            // Contraintes : toutes différentes par bloc 3x3
            for box_row in 0..3 {
                for box_col in 0..3 {
                    let mut vars: Vec<Int> = Vec::new();
                    for i in 0..3 {
                        for j in 0..3 {
                            vars.push(x[box_row * 3 + i][box_col * 3 + j].clone());
                        }
                    }
                    solver.assert(&Int::distinct(&vars));
                }
            }

            // Résoudre
            if solver.check() == SatResult::Sat {
                let model = match solver.get_model() {
                    Some(m) => m,
                    None => return false,
                };
                for i in 0..9 {
                    for j in 0..9 {
                        // Une valeur absente du modèle interrompt la résolution
                        // (false) : jamais de panique.
                        match model.eval(&x[i][j], true).and_then(|v| v.as_i64()) {
                            Some(val) => grid.cells[i][j] = val as u8,
                            None => return false,
                        }
                    }
                }
                return true;
            }

            false
        }
    }

    // Test
    let dir = Path::new("Puzzles");
    let hard_puzzles = match charger_puzzles(&dir.join("Sudoku_hardest.txt"), None) {
        Ok(v) => v,
        Err(e) => {
            println!("Lecture des puzzles difficiles impossible : {}", e);
            Vec::new()
        }
    };
    let premier = match hard_puzzles.first() {
        Some(p) => p.clone(),
        None => {
            println!("Aucun puzzle difficile disponible");
            return;
        }
    };

    let z3_solver = Z3Solver;
    let mut test_grid = match SudokuGrid::from_string(&premier) {
        Some(g) => g,
        None => {
            println!("Grille illisible");
            return;
        }
    };
    println!("Puzzle difficile:");
    println!("{}", test_grid.render());

    let start = Instant::now();
    let solved = z3_solver.solve(&mut test_grid);
    let elapsed = start.elapsed().as_secs_f64() * 1000.0;

    println!(
        "\nRésolu: {} en {:.2} ms",
        if solved { "True" } else { "False" },
        elapsed
    );
    println!("\nSolution:");
    println!("{}", test_grid.render());
};


Puzzle difficile:


8 5 . | . . 2 | 4 . . 


7 2 . | . . . | . . 9 


. . 4 | . . . | . . . 


---------------------


. . . | 1 . 7 | . . 2 


... (15 outputs left out)

---------------------


4 3 2 | 9 8 1 | 6 7 5 


6 1 7 | 4 2 5 | 8 9 3 


5 9 8 | 7 3 6 | 2 4 1 


### Interprétation : résultat Z3 Int

Le solveur Z3 avec des entiers a résolu le puzzle difficile (durée mesurée dynamiquement par la cellule de code ci-dessous).

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| **Statut** | sat | Solution trouvée |
| **Type de variables** | Int | Entiers symboliques |

**Points clés** :
1. **API simple** : la contrainte `Distinct` est très expressive
2. **Théorie des entiers** : Z3 utilise QF_LIA (Linear Integer Arithmetic)
3. **Flexibilité** : Z3 peut facilement étendre le modèle avec d'autres contraintes

> **Note technique** : la contrainte `Distinct` de Z3 est plus puissante que l'inégalité binaire car elle permet au solveur d'utiliser des algorithmes spécialisés pour la propagation.
>
> **Les durées ne sont pas figées** : le temps de résolution (machine-dépendant) est mesuré dynamiquement par la cellule de code ci-dessous ; il n'est écrit nulle part dans ce carnet.


### Exercice : contrainte de somme sur un bloc (Killer Sudoku)

Le solveur Z3 avec `Int` permet d'ajouter facilement des contraintes arithmétiques. Le **Killer Sudoku** est une variante où des groupes de cellules doivent satisfaire une somme cible. Implémentez un solveur qui ajoute une contrainte de somme sur un bloc 3x3 spécifique. La somme naturelle d'un bloc est 1+2+...+9 = 45.

En Rust, la somme s'écrit `Int::add(&[&terme_1, &terme_2, ...])` : on construit une expression `Int` par terme, puis on les additionne.


In [6]:
{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }

    /// Résout un Sudoku avec une contrainte supplémentaire de somme sur un bloc 3x3.
    ///
    /// Cette variante s'inspire du "Killer Sudoku" où des groupes de cellules
    /// doivent satisfaire une contrainte arithmétique en plus des règles classiques.
    ///
    /// La fonction rend `None` tant que l'étudiant n'a pas implémenté la
    /// résolution : aucune erreur, aucun panic (règle C.1).
    fn solve_with_sum_constraint(
        grid: &mut SudokuGrid,
        block_row: usize,
        block_col: usize,
        target_sum: i64,
    ) -> Option<bool> {
        // TODO etudiant : implémenter la contrainte de somme sur un bloc
        // Etape 1 : créer le Solver Z3 et les 81 variables Int (domaine [1,9])
        // Etape 2 : ajouter les 27 contraintes Distinct (lignes, colonnes, blocs)
        // Etape 3 : fixer les valeurs initiales de la grille
        // Etape 4 : calculer les coordonnées du bloc (block_row * 3 + i, block_col * 3 + j)
        // Etape 5 : ajouter la contrainte de somme des cellules du bloc == target_sum
        // Etape 6 : résoudre et mettre à jour la grille si solution trouvée
        let _ = (grid, block_row, block_col, target_sum);
        None // TODO etudiant : remplacer par la résolution réelle
    }

    println!("Exercice a completer : contrainte de somme sur bloc");
};


Exercice a completer : contrainte de somme sur bloc


Passage au solveur BitVector (objectif 4 : comparer les deux encodages).


In [7]:
use std::path::Path;
use std::time::Instant;
use z3::ast::{Ast, BV, Int};
use z3::{SatResult, Solver};

{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }
    /// Lecture des puzzles : l'erreur d'ouverture est TRAITÉE (`Result`),
    /// jamais avalée.
    fn charger_puzzles(chemin: &std::path::Path, max_puzzles: Option<usize>) -> Result<Vec<String>, std::io::Error> {
        let contenu = std::fs::read_to_string(chemin)?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect::<String>());
                if let Some(max) = max_puzzles {
                    if puzzles.len() >= max {
                        break;
                    }
                }
            }
        }
        Ok(puzzles)
    }

    /// Solveur Sudoku utilisant des BitVectors 4 bits.
    struct Z3BitVectorSolver;

    /// Ajoute les inégalités par paires : sur un BitVector, `Distinct`
    /// n'est pas disponible, on décompose donc la contrainte.
    fn ajouter_toutes_differentes(solver: &mut Solver, cellules: &[BV]) {
        for i in 0..cellules.len() {
            for j in (i + 1)..cellules.len() {
                solver.assert(&cellules[i]._eq(&cellules[j]).not());
            }
        }
    }

    impl Z3BitVectorSolver {
        /// Résout avec BitVectors.
        fn solve(&self, grid: &mut SudokuGrid) -> bool {
            // BitVectors 4 bits (les valeurs 1-9 tiennent dans 4 bits)
            let mut x: Vec<Vec<BV>> = Vec::new();
            for i in 0..9 {
                let mut ligne: Vec<BV> = Vec::new();
                for j in 0..9 {
                    ligne.push(BV::new_const(format!("x_{i}_{j}"), 4));
                }
                x.push(ligne);
            }

            let mut solver = Solver::new();

            // Contraintes de domaine (comparaisons non signées)
            for i in 0..9 {
                for j in 0..9 {
                    solver.assert(&x[i][j].bvuge(&BV::from_i64(1, 4)));
                    solver.assert(&x[i][j].bvule(&BV::from_i64(9, 4)));
                }
            }

            // Valeurs initiales
            for i in 0..9 {
                for j in 0..9 {
                    if grid.cells[i][j] != 0 {
                        solver.assert(&x[i][j]._eq(&BV::from_i64(grid.cells[i][j] as i64, 4)));
                    }
                }
            }

            // Lignes
            for i in 0..9 {
                let cellules: Vec<BV> = (0..9).map(|j| x[i][j].clone()).collect();
                ajouter_toutes_differentes(&mut solver, &cellules);
            }

            // Colonnes
            for j in 0..9 {
                let cellules: Vec<BV> = (0..9).map(|i| x[i][j].clone()).collect();
                ajouter_toutes_differentes(&mut solver, &cellules);
            }

            // Blocs
            for box_row in 0..3 {
                for box_col in 0..3 {
                    let mut cellules: Vec<BV> = Vec::new();
                    for i in 0..3 {
                        for j in 0..3 {
                            cellules.push(x[box_row * 3 + i][box_col * 3 + j].clone());
                        }
                    }
                    ajouter_toutes_differentes(&mut solver, &cellules);
                }
            }

            if solver.check() == SatResult::Sat {
                let model = match solver.get_model() {
                    Some(m) => m,
                    None => return false,
                };
                for i in 0..9 {
                    for j in 0..9 {
                        match model.eval(&x[i][j], true).and_then(|v| v.as_u64()) {
                            Some(val) => grid.cells[i][j] = val as u8,
                            None => return false,
                        }
                    }
                }
                return true;
            }

            false
        }
    }

    /// Solveur Z3 Int, reconstruit pour la comparaison (même encodage que la
    /// section précédente, solveur reconstruit dans la cellule).
    struct Z3Solver;

    impl Z3Solver {
        fn solve(&self, grid: &mut SudokuGrid) -> bool {
            let mut x: Vec<Vec<Int>> = Vec::new();
            for i in 0..9 {
                let mut ligne: Vec<Int> = Vec::new();
                for j in 0..9 {
                    ligne.push(Int::new_const(format!("x_{i}_{j}")));
                }
                x.push(ligne);
            }

            let mut solver = Solver::new();

            for i in 0..9 {
                for j in 0..9 {
                    solver.assert(&x[i][j].ge(&Int::from_i64(1)));
                    solver.assert(&x[i][j].le(&Int::from_i64(9)));
                    if grid.cells[i][j] != 0 {
                        solver.assert(&x[i][j]._eq(&Int::from_i64(grid.cells[i][j] as i64)));
                    }
                }
            }

            for i in 0..9 {
                let vars: Vec<Int> = (0..9).map(|j| x[i][j].clone()).collect();
                solver.assert(&Int::distinct(&vars));
            }
            for j in 0..9 {
                let vars: Vec<Int> = (0..9).map(|i| x[i][j].clone()).collect();
                solver.assert(&Int::distinct(&vars));
            }
            for box_row in 0..3 {
                for box_col in 0..3 {
                    let mut vars: Vec<Int> = Vec::new();
                    for i in 0..3 {
                        for j in 0..3 {
                            vars.push(x[box_row * 3 + i][box_col * 3 + j].clone());
                        }
                    }
                    solver.assert(&Int::distinct(&vars));
                }
            }

            if solver.check() == SatResult::Sat {
                let model = match solver.get_model() {
                    Some(m) => m,
                    None => return false,
                };
                for i in 0..9 {
                    for j in 0..9 {
                        match model.eval(&x[i][j], true).and_then(|v| v.as_i64()) {
                            Some(val) => grid.cells[i][j] = val as u8,
                            None => return false,
                        }
                    }
                }
                return true;
            }

            false
        }
    }

    // Comparaison Z3 Int vs BitVec
    println!("=== Comparaison Z3 Int vs BitVector ===");

    let dir = Path::new("Puzzles");
    let hard_puzzles = match charger_puzzles(&dir.join("Sudoku_hardest.txt"), None) {
        Ok(v) => v,
        Err(e) => {
            println!("Lecture des puzzles difficiles impossible : {}", e);
            Vec::new()
        }
    };

    let z3_int = Z3Solver;
    let z3_bv = Z3BitVectorSolver;

    for (i, puzzle_str) in hard_puzzles.iter().take(5).enumerate() {
        println!("\nPuzzle {}:", i + 1);

        let mut grid1 = match SudokuGrid::from_string(puzzle_str) {
            Some(g) => g,
            None => continue,
        };
        let start = Instant::now();
        z3_int.solve(&mut grid1);
        let t1 = start.elapsed().as_secs_f64() * 1000.0;

        let mut grid2 = match SudokuGrid::from_string(puzzle_str) {
            Some(g) => g,
            None => continue,
        };
        let start = Instant::now();
        z3_bv.solve(&mut grid2);
        let t2 = start.elapsed().as_secs_f64() * 1000.0;

        println!("  Z3 Int:      {:>8.2} ms", t1);
        println!("  Z3 BitVec:   {:>8.2} ms", t2);
    }
};


=== Comparaison Z3 Int vs BitVector ===


Puzzle 1:


  Z3 Int:        156.54 ms


  Z3 BitVec:      41.28 ms


Puzzle 2:


  Z3 Int:        283.57 ms


... (6 outputs left out)

  Z3 BitVec:      28.59 ms


Puzzle 5:


  Z3 Int:         72.37 ms


  Z3 BitVec:      37.22 ms


### Interprétation : Z3 Int vs BitVector

La comparaison porte sur deux encodages du même problème : les entiers symboliques et les vecteurs de bits. Sur le jumeau Python, l'écart mesuré est de l'ordre de **2x à 6x selon le puzzle** en faveur des BitVectors — un ordre de grandeur stable d'une machine à l'autre, dont les valeurs exactes dépendent du matériel. La cellule de benchmark ci-dessous remesure ce rapport sur ce portage : l'ordre de grandeur se lit dans sa sortie, les valeurs exactes ne sont figurées nulle part.

**Points clés** :
1. **BitVectors plus proches du hardware** : Z3 peut utiliser des opérations au niveau du bit
2. **Domaine restreint** : 4 bits = 16 valeurs, exactement ce qu'il faut pour 1-9
3. **Propagation plus efficace** : les contraintes au niveau du bit sont plus simples à propager

> **Note technique** : les BitVectors sont représentés en précision fixe (4 bits ici), ce qui permet à Z3 d'utiliser des algorithmes de propagation plus efficaces. Les entiers symboliques en Z3 ont une précision arbitraire, ce qui ajoute de la complexité.
>
> **Le coût de l'encodage** : sur les BitVectors, la contrainte `Distinct` du jumeau Python se décompose en inégalités par paires — c'est le prix de la représentation compacte.


## 4. Résumé et conclusions

Ce notebook a présenté deux approches Z3 pour résoudre le Sudoku : avec des entiers symboliques et avec des BitVectors.

### Comparaison des approches Z3

| Approche | Type | Avantages | Inconvénients |
|----------|------|-----------|---------------|
| **Z3 Int** | Entiers symboliques | API simple, contrainte `Distinct` expressive | Plus lent |
| **Z3 BitVec** | BitVectors 4 bits | Plus rapide, proche du hardware | Syntaxe plus complexe |

### Recommandations

- **Simplicité** : Z3 Int avec la contrainte `Distinct`
- **Performance** : Z3 BitVec pour les domaines restreints
- **Flexibilité** : Z3 supporte de nombreuses théories (arithmétique, tableaux, etc.)

### Au-delà du Sudoku

Z3 peut résoudre une grande variété de problèmes :
- Vérification de programmes
- Analyse de sécurité
- Synthèse de code
- Théorèmes de logique

Le Sudoku est un excellent problème pédagogique pour comprendre la différence entre SAT (propositionnel) et SMT (théories).


## 5. Exercices et exemples guidés

Cette section contient :

- **Exercice 1** : compter le nombre de solutions d'une grille donnée (à implémenter)
- **Exemple guidé** : solveur Z3 avec contrainte diagonale (solution fournie pour référence)
- **Exercice 2** : coloration de graphe avec Z3 (à implémenter)
- **Exemple guidé** : optimisation SMT avec `Optimize` / `maximize` (carré latin à reward maximal, solution complète)
- **Exercice 3** : minimiser un coût diagonal avec `Optimize` (à implémenter)
- **Conclusion** : bilan comparatif Int vs BitVec et ouverture SMT

Chaque bloc est clairement marqué comme "exercice" (à compléter) ou "exemple" (solution complète).


### Exercice 1 : Compter le nombre de solutions

Implémentez un solveur Z3 qui compte le nombre total de solutions d'une grille de Sudoku.

**Indices** :
- Utiliser `Solver::new()` et ajouter les contraintes classiques
- Après chaque `solver.check() == SatResult::Sat`, ajouter une contrainte excluant le modèle trouvé (`solver.assert(&Bool::or(&[...]))`, une inégalité par case)
- Compter les itérations jusqu'à `SatResult::Unsat`


In [8]:
use std::path::Path;

{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }
    /// Lecture des puzzles : l'erreur d'ouverture est TRAITÉE (`Result`),
    /// jamais avalée.
    fn charger_puzzles(chemin: &std::path::Path, max_puzzles: Option<usize>) -> Result<Vec<String>, std::io::Error> {
        let contenu = std::fs::read_to_string(chemin)?;
        let mut puzzles: Vec<String> = Vec::new();
        for ligne in contenu.lines() {
            let l = ligne.trim();
            if l.chars().count() >= 81 {
                puzzles.push(l.chars().take(81).collect::<String>());
                if let Some(max) = max_puzzles {
                    if puzzles.len() >= max {
                        break;
                    }
                }
            }
        }
        Ok(puzzles)
    }

    /// Solveur Z3 qui compte les solutions jusqu'à un maximum.
    struct Z3SolutionCounter;

    impl Z3SolutionCounter {
        /// Compte le nombre de solutions jusqu'à `max_count`.
        ///
        /// Rend `None` tant que l'étudiant n'a pas implémenté le compteur :
        /// aucune erreur, aucun panic (règle C.1).
        fn count_solutions(&self, grid: &SudokuGrid, max_count: usize) -> Option<usize> {
            // TODO etudiant : implémenter le compteur
            // 1. Créer le Solver et les variables Z3 (Int)
            // 2. Ajouter les contraintes Sudoku (domaine, valeurs fixées, lignes/colonnes/blocs)
            // 3. Boucle : check(), bloquer la solution trouvée, incrémenter le compteur
            let _ = (grid, max_count);
            None
        }
    }

    // Test
    let counter = Z3SolutionCounter;
    let dir = Path::new("Puzzles");
    let mut resultat: Option<usize> = None;
    if let Ok(easy_puzzles) = charger_puzzles(&dir.join("Sudoku_Easy51.txt"), Some(10)) {
        if let Some(premier) = easy_puzzles.first() {
            if let Some(grille) = SudokuGrid::from_string(premier) {
                resultat = counter.count_solutions(&grille, 10);
            }
        }
    }
    match resultat {
        Some(n) => println!("Nombre de solutions (max 10): {}", n),
        None => println!("Implementation a completer pour voir le resultat"),
    }
};


Implementation a completer pour voir le resultat


### Exemple guidé : solveur Z3 avec contrainte diagonale

L'implémentation ci-dessous étend le solveur Z3 classique en ajoutant les contraintes `Distinct` sur les deux diagonales principales.


In [9]:
use z3::ast::{Ast, Int};
use z3::{SatResult, Solver};

{
    #[derive(Clone)]
    struct SudokuGrid {
        cells: [[u8; 9]; 9],
    }

    impl SudokuGrid {
        /// Construit une grille depuis 81 caractères ('.' ou '0' = case vide).
        /// Une chaîne mal formée rend `None` : jamais de panique.
        fn from_string(s: &str) -> Option<SudokuGrid> {
            // '.' vaut 0 (case vide) ; les espaces et sauts de ligne sont
            // retires. La longueur utile reste donc de 81 caractères.
            let brut: Vec<char> = s
                .chars()
                .filter(|c| *c != ' ' && *c != '\n')
                .map(|c| if c == '.' { '0' } else { c })
                .collect();
            if brut.len() != 81 {
                return None;
            }
            let mut grid = SudokuGrid { cells: [[0u8; 9]; 9] };
            for i in 0..81 {
                grid.cells[i / 9][i % 9] = brut[i].to_digit(10)? as u8;
            }
            Some(grid)
        }

        /// Reproduit exactement la mise en forme du jumeau Python :
        /// séparateur de 21 tirets tous les 3 lignes, '| ' tous les 3 colonnes,
        /// '.' pour une case vide.
        fn render(&self) -> String {
            let mut lignes: Vec<String> = Vec::new();
            for r in 0..9 {
                if r > 0 && r % 3 == 0 {
                    lignes.push("-".repeat(21));
                }
                let mut ligne = String::new();
                for c in 0..9 {
                    if c > 0 && c % 3 == 0 {
                        ligne.push_str("| ");
                    }
                    let val = self.cells[r][c];
                    if val != 0 {
                        ligne.push_str(&val.to_string());
                    } else {
                        ligne.push('.');
                    }
                    ligne.push(' ');
                }
                lignes.push(ligne);
            }
            lignes.join("\n")
        }
    }

    /// Solveur Z3 pour Sudoku avec contraintes diagonales supplémentaires.
    struct Z3DiagonalSolver;

    impl Z3DiagonalSolver {
        /// Résout le Sudoku diagonal avec Z3.
        fn solve(&self, grid: &mut SudokuGrid) -> bool {
            // Contraintes diagonales
            let mut s = Solver::new();

            // Variables
            let mut cells: Vec<Vec<Int>> = Vec::new();
            for r in 0..9 {
                let mut ligne: Vec<Int> = Vec::new();
                for c in 0..9 {
                    ligne.push(Int::new_const(format!("c_{r}_{c}")));
                }
                cells.push(ligne);
            }

            // Domaine [1..9]
            for r in 0..9 {
                for c in 0..9 {
                    s.assert(&cells[r][c].ge(&Int::from_i64(1)));
                    s.assert(&cells[r][c].le(&Int::from_i64(9)));
                }
            }

            // Valeurs fixées
            for r in 0..9 {
                for c in 0..9 {
                    if grid.cells[r][c] != 0 {
                        s.assert(&cells[r][c]._eq(&Int::from_i64(grid.cells[r][c] as i64)));
                    }
                }
            }

            // Contraintes classiques : lignes, colonnes, blocs
            for i in 0..9 {
                let ligne: Vec<Int> = (0..9).map(|c| cells[i][c].clone()).collect();
                s.assert(&Int::distinct(&ligne));
                let colonne: Vec<Int> = (0..9).map(|r| cells[r][i].clone()).collect();
                s.assert(&Int::distinct(&colonne));
            }

            for br in 0..3 {
                for bc in 0..3 {
                    let mut bloc: Vec<Int> = Vec::new();
                    for r in 0..3 {
                        for c in 0..3 {
                            bloc.push(cells[br * 3 + r][bc * 3 + c].clone());
                        }
                    }
                    s.assert(&Int::distinct(&bloc));
                }
            }

            // Contraintes diagonales
            let principale: Vec<Int> = (0..9).map(|i| cells[i][i].clone()).collect();
            s.assert(&Int::distinct(&principale));
            let secondaire: Vec<Int> = (0..9).map(|i| cells[i][8 - i].clone()).collect();
            s.assert(&Int::distinct(&secondaire));

            // Résolution
            if s.check() == SatResult::Sat {
                let m = match s.get_model() {
                    Some(model) => model,
                    None => return false,
                };
                for r in 0..9 {
                    for c in 0..9 {
                        match m.eval(&cells[r][c], true).and_then(|v| v.as_i64()) {
                            Some(val) => grid.cells[r][c] = val as u8,
                            None => return false,
                        }
                    }
                }
                return true;
            }
            false
        }
    }

    /// Met en forme une diagonale comme une liste Python : [1, 4, 3, ...].
    fn formater_liste(valeurs: &[u8]) -> String {
        let morceaux: Vec<String> = valeurs.iter().map(|v| v.to_string()).collect();
        format!("[{}]", morceaux.join(", "))
    }

    /// Une diagonale est valide si ses 9 valeurs sont toutes distinctes.
    fn tous_distincts(valeurs: &[u8]) -> bool {
        let mut tri: Vec<u8> = valeurs.to_vec();
        tri.sort();
        tri.dedup();
        tri.len() == 9
    }

    // Test avec une grille diagonale
    let diagonal_solver = Z3DiagonalSolver;

    let test = "100000000000003000080005020231000000060000000097000231000978000000000000078000000";

    let mut diag_grid = match SudokuGrid::from_string(test) {
        Some(g) => g,
        None => {
            println!("Grille illisible");
            return;
        }
    };
    println!("Grille avant résolution :");
    println!("{}", diag_grid.render());
    println!();

    if diagonal_solver.solve(&mut diag_grid) {
        println!("Solution trouvée :");
        println!("{}", diag_grid.render());
        // Vérification des diagonales
        let diag1: Vec<u8> = (0..9).map(|i| diag_grid.cells[i][i]).collect();
        let diag2: Vec<u8> = (0..9).map(|i| diag_grid.cells[i][8 - i]).collect();
        println!(
            "\nDiagonale principale : {} — tous distincts : {}",
            formater_liste(&diag1),
            if tous_distincts(&diag1) { "True" } else { "False" }
        );
        println!(
            "Diagonale secondaire : {} — tous distincts : {}",
            formater_liste(&diag2),
            if tous_distincts(&diag2) { "True" } else { "False" }
        );
    } else {
        println!("Pas de solution");
    }
};


Grille avant résolution :


1 . . | . . . | . . . 


. . . | . . 3 | . . . 


. 8 . | . . 5 | . 2 . 


---------------------


2 3 1 | . . . | . . . 


... (16 outputs left out)

4 1 9 | 5 3 6 | 7 8 2 


3 7 8 | 4 1 2 | 5 6 9 


Diagonale principale : [1, 5, 6, 7, 2, 4, 3, 8, 9] — tous distincts : True


Diagonale secondaire : [8, 7, 4, 9, 2, 6, 5, 1, 3] — tous distincts : True


### Lecture du solveur diagonal : deux contraintes de plus, rien d'autre

Comparez les deux grilles affichées : la grille d'entrée est **clairsemée** (une vingtaine d'indices), et la solution trouvée vérifie les **deux diagonales toutes-distinctes**. C'est la variante **Sudoku-X**, et la leçon est dans le code : passer du Sudoku standard au Sudoku-X a coûté exactement **deux lignes** — deux `Distinct` supplémentaires sur les diagonales `cells[i][i]` et `cells[i][8-i]`. Aucune heuristique à ré-accorder, aucun solveur à ré-écrire : c'est l'**additivité déclarative** du SMT, la propriété qui le distingue d'un solveur dédié.

Une précaution de lecture : cette grille de test admet **plusieurs solutions** (le blocage itératif du modèle en compte au moins trois côté jumeau). Les valeurs de diagonale affichées par la cellule dépendent donc du modèle que Z3 retient — elles sont un témoin parmi d'autres, pas une valeur unique. Ce qui est stable, c'est l'invariant : **les deux diagonales sont toutes distinctes**. La ligne « tous distincts : True » est l'attendu robuste ; la liste de valeurs est l'attendu modèle-dépendant.


### Exercice 2 : coloration de graphe avec Z3

Z3 n'est pas limité au Sudoku ! Il peut résoudre de nombreux problèmes de satisfaction de contraintes.

**Objectif** : implémenter un solveur de **coloration de graphe** (graph coloring) avec Z3.

Le problème consiste à colorier une carte de régions avec un nombre minimum de couleurs,
de sorte que deux régions voisines (partageant une frontière) n'aient jamais la même couleur.

**Théorème des 4 couleurs** : toute carte plane peut être coloriée avec au plus 4 couleurs.

**Étapes** :
1. Modéliser chaque région par une variable entière `Int` représentant sa couleur (0 à 3)
2. Ajouter les contraintes de domaine : `0 <= couleur <= 3`
3. Pour chaque paire de régions voisines, ajouter la contrainte `couleur_i != couleur_j`
4. Résoudre avec `solver.check() == SatResult::Sat` et extraire la coloration

> **Code à compléter dans la cellule suivante**


In [10]:
use std::collections::HashMap;

{
    /// Résout le problème de coloration de graphe avec Z3.
    ///
    /// Le graphe représente une partie simplifiée de la carte de France
    /// avec 10 régions et leurs voisinages.
    ///
    /// Rend `None` tant que l'étudiant n'a pas implémenté la modélisation :
    /// aucune erreur, aucun panic (règle C.1).
    fn solve_graph_coloring() -> Option<HashMap<String, usize>> {
        // Graphe : région -> voisins
        let regions: Vec<(&str, Vec<&str>)> = vec![
            ("Ile-de-France", vec!["Picardie", "Normandie", "Centre", "Bourgogne"]),
            ("Picardie", vec!["Ile-de-France", "Normandie", "Champagne"]),
            ("Normandie", vec!["Ile-de-France", "Picardie", "Bretagne", "Pays-de-la-Loire", "Centre"]),
            ("Bretagne", vec!["Normandie", "Pays-de-la-Loire"]),
            ("Pays-de-la-Loire", vec!["Bretagne", "Normandie", "Centre", "Aquitaine"]),
            ("Centre", vec!["Ile-de-France", "Normandie", "Pays-de-la-Loire", "Aquitaine", "Bourgogne", "Auvergne"]),
            ("Bourgogne", vec!["Ile-de-France", "Centre", "Champagne", "Auvergne"]),
            ("Champagne", vec!["Picardie", "Bourgogne", "Auvergne"]),
            ("Auvergne", vec!["Centre", "Bourgogne", "Champagne", "Aquitaine"]),
            ("Aquitaine", vec!["Pays-de-la-Loire", "Centre", "Auvergne"]),
        ];
        let num_colors = 4;
        let color_names = ["Bleu", "Blanc", "Rouge", "Vert"];
        let _ = (num_colors, &color_names, &regions);

        // TODO etudiant : modéliser le problème avec Z3
        // 1. Créer une variable Int pour chaque région
        // 2. Ajouter les contraintes de domaine [0, num_colors - 1]
        // 3. Pour chaque paire de régions voisines, ajouter color_i != color_j
        // 4. Résoudre et retourner le résultat
        None
    }

    // Test
    match solve_graph_coloring() {
        Some(coloration) => {
            println!("Coloration trouvee :");
            let mut lignes: Vec<(&String, &usize)> = coloration.iter().collect();
            lignes.sort_by(|a, b| a.0.cmp(b.0));
            for (region, color) in lignes {
                println!("  {}: {}", region, color);
            }
        }
        None => println!("Pas de solution trouvee"),
    }
};


Pas de solution trouvee


### Lecture honnête : « Pas de solution trouvee » = le stub rendu proprement

La ligne unique de la sortie est la sortie **conventionnelle d'un exercice incomplet** (règle C.1) : la fonction rend `None` proprement au lieu de lever une erreur, et l'énoncé imprime le diagnostic. Le problème attendu — colorier la carte de France à 10 régions avec 4 couleurs via `Int(color_i)` et `color_i != color_j` par arête — est exactement la modélisation du Sudoku transposée : des variables à domaine fini, des contraintes de différence binaire. Le théorème des quatre couleurs garantit qu'une solution existe pour toute carte planaire ; l'exercice, lui, attend sa modélisation.


### Exemple guidé : optimisation SMT avec Z3 (`Optimize` / `maximize`)

Jusqu'ici, Z3 a résolu des problèmes de **satisfaction** : trouver UNE solution
réalisable (une grille de Sudoku valide) via `Solver` + `check` + `get_model`.

La capacité distinctive de Z3 est l'**optimisation SMT** : le contexte
`Optimize` résout des problèmes où l'on **maximise** (ou **minimise**) un
objectif sous contraintes — c'est le moteur de la programmation **MaxSMT**
(maximisation sat-modulo-théories, utilisé en vérification, scheduling,
allocation de ressources, debug de modèles).

**Démonstration** : un « carré latin à bonus » — chaque cellule (i, j) offre un
*reward* dépendant de la valeur placée. Parmi tous les carrés latins 5x5
valides, `Optimize` + `maximize` trouve celui qui **maximise le reward
total**. Le statut `sat` certifie que l'optimum est atteint (preuve
non-vacuous).

> Miroir de la démo CP-SAT du carnet OR-Tools de la série (même problème, même
> reward 178), mais avec l'API SMT `Optimize` au lieu de `maximize` du
> modèle CP-SAT.

La **matrice de récompenses est figée** dans la cellule : elle a été relevée sur
le jumeau Python (`random.Random(42)` puis `randint(0, 10)`) et reportée en
données littérales, pour que le reward maximal reste un attendu reproductible
d'un moteur à l'autre.


In [11]:
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult};

{
    /// Matrice de récompenses FIGÉE : relevée sur le jumeau Python par
    /// `random.Random(42).randint(0, 10)`, reportée en données littérales
    /// pour que le reward maximal reste reproductible.
    let reward: Vec<Vec<Vec<i64>>> = vec![
        vec![vec![10, 1, 0, 4, 3, 3], vec![2, 1, 10, 8, 1, 9], vec![6, 0, 0, 1, 3, 3], vec![8, 9, 0, 8, 3, 10], vec![8, 6, 3, 7, 9, 4]],
        vec![vec![0, 2, 6, 5, 4, 2], vec![3, 5, 1, 1, 6, 1], vec![5, 5, 9, 4, 0, 7], vec![8, 1, 6, 1, 8, 4], vec![10, 9, 5, 9, 3, 1]],
        vec![vec![0, 10, 3, 4, 1, 3], vec![1, 6, 4, 7, 10, 5], vec![2, 5, 5, 3, 10, 4], vec![10, 10, 1, 9, 10, 2], vec![8, 3, 2, 7, 6, 4]],
        vec![vec![10, 8, 3, 10, 5, 0], vec![3, 0, 5, 6, 4, 1], vec![3, 9, 5, 3, 10, 7], vec![6, 10, 7, 2, 4, 2], vec![3, 8, 8, 4, 9, 6]],
        vec![vec![9, 6, 5, 3, 2, 8], vec![7, 1, 0, 1, 2, 10], vec![2, 10, 6, 9, 1, 6], vec![6, 9, 7, 8, 4, 8], vec![0, 10, 1, 10, 8, 4]],
    ];

    /// Trouve le carré latin d'ordre `n` qui MAXIMISE le reward total.
    ///
    /// reward[i][j][v] = bonus pour placer la valeur v (1..n) dans la case (i, j).
    /// On utilise le contexte d'optimisation `Optimize` avec `maximize`.
    /// Rend l'optimum, la valeur de l'objectif et le statut.
    fn solve_max_reward_latin_square_z3(
        n: usize,
        reward: &Vec<Vec<Vec<i64>>>,
    ) -> Option<(Vec<Vec<usize>>, i64, SatResult)> {
        let mut opt = Optimize::new();
        let mut x: Vec<Vec<Vec<Bool>>> = Vec::new();
        for i in 0..n {
            let mut ligne: Vec<Vec<Bool>> = Vec::new();
            for j in 0..n {
                let mut cellule: Vec<Bool> = Vec::new();
                for v in 1..=n {
                    cellule.push(Bool::new_const(format!("x_{i}_{j}_{v}")));
                }
                // exactement une valeur par case
                let paires: Vec<(&Bool, i32)> = cellule.iter().map(|b| (b, 1)).collect();
                opt.assert(&Bool::pb_eq(&paires, 1));
                ligne.push(cellule);
            }
            x.push(ligne);
        }
        // chaque valeur une fois par ligne
        for i in 0..n {
            for v in 1..=n {
                let paires: Vec<(&Bool, i32)> = (0..n).map(|j| (&x[i][j][v - 1], 1)).collect();
                opt.assert(&Bool::pb_eq(&paires, 1));
            }
        }
        // chaque valeur une fois par colonne
        for j in 0..n {
            for v in 1..=n {
                let paires: Vec<(&Bool, i32)> = (0..n).map(|i| (&x[i][j][v - 1], 1)).collect();
                opt.assert(&Bool::pb_eq(&paires, 1));
            }
        }

        // objectif : maximiser le reward total
        let mut termes: Vec<Int> = Vec::new();
        for i in 0..n {
            for j in 0..n {
                for v in 1..=n {
                    termes.push(
                        x[i][j][v - 1].ite(&Int::from_i64(reward[i][j][v]), &Int::from_i64(0)),
                    );
                }
            }
        }
        let objectif = Int::add(&termes);
        opt.maximize(&objectif);

        let status = opt.check(&[]);
        if status != SatResult::Sat {
            return None;
        }
        let model = match opt.get_model() {
            Some(m) => m,
            None => return None,
        };
        let mut grid: Vec<Vec<usize>> = vec![vec![0usize; n]; n];
        for i in 0..n {
            for j in 0..n {
                for v in 1..=n {
                    let vrai = model
                        .eval(&x[i][j][v - 1], true)
                        .and_then(|a| a.as_bool())
                        .unwrap_or(false);
                    if vrai {
                        grid[i][j] = v;
                    }
                }
            }
        }
        // Objectif recalculé sur la grille : une valeur numérique propre.
        let mut computed: i64 = 0;
        for i in 0..n {
            for j in 0..n {
                computed += reward[i][j][grid[i][j]];
            }
        }
        Some((grid, computed, status))
    }

    match solve_max_reward_latin_square_z3(5, &reward) {
        Some((grid, obj, status)) => {
            let libelle = match status {
                SatResult::Sat => "sat",
                SatResult::Unsat => "unsat",
                SatResult::Unknown => "unknown",
            };
            println!("Statut Z3 Optimize : {}", libelle);
            println!("Reward maximal     : {}", obj);
            println!("Carre latin optimal (max reward) :");
            for row in &grid {
                let ligne: Vec<String> = row.iter().map(|v| format!("{:2}", v)).collect();
                println!("{}", ligne.join(" "));
            }
            let n = grid.len();
            let attendu: Vec<usize> = (1..=n).collect();
            let lignes_valides = grid.iter().all(|row| {
                let mut tri = row.clone();
                tri.sort();
                tri == attendu
            });
            let colonnes_valides = (0..n).all(|j| {
                let mut tri: Vec<usize> = (0..n).map(|i| grid[i][j]).collect();
                tri.sort();
                tri == attendu
            });
            println!(
                "Carre latin valide : {}",
                if lignes_valides && colonnes_valides { "True" } else { "False" }
            );
        }
        None => println!("Statut Z3 Optimize : inconnu"),
    }
};


Statut Z3 Optimize : sat


Reward maximal     : 178


Carre latin optimal (max reward) :


 3  2  1  5  4


 2  1  5  4  3


 1  4  2  3  5


 5  3  4  1  2


 4  5  3  2  1


Carre latin valide : True


### Lecture du `Optimize` : sat, reward 178, et la preuve d'optimalité

Quatre lignes, quatre enseignements. **`Statut : sat`** — le contexte `Optimize` de Z3 est d'abord un solveur de satisfaction. **`Reward maximal : 178`** — ce n'est pas un reward *trouvé*, c'est un reward **prouvé maximal** : `Optimize` boucle en résolvant puis en ajoutant la contrainte « objectif > meilleur courant » jusqu'à l'insatisfaisabilité — la différence entre « une bonne solution » et « la meilleure » est exactement la différence entre `Solver` et `Optimize`. **Le carré latin 5x5 affiché** — chaque ligne et chaque colonne est une permutation de 1-5, vérifié par la ligne « Carré latin valide : True » (citation de la sortie). La docstring de la cellule le dit : cette démo est le miroir SMT de l'exemple CP-SAT de la série OR-Tools — deux moteurs, une même capacité signature, la maximisation sous contraintes.

Une nuance de lecture : l'optimum est **unique en valeur** (178), mais pas nécessairement en solution — plusieurs carrés latins peuvent atteindre 178. Le carré affiché est donc l'un des optima, ce qui n'affecte pas la valeur de l'objectif.


### Exercice : minimiser un coût diagonal avec Z3 `Optimize`

Adaptez la modélisation ci-dessus pour **MINIMISER** une fonction de coût au
lieu de maximiser le reward.

**Indices** :
1. Gardez les mêmes contraintes (carré latin : `Bool::pb_eq` one-value-per-cell,
   latin rows/cols).
2. Remplacez `opt.maximize(&objectif)` par :
   ```rust
   let diagonal_sum = Int::add(&[ /* somme des valeurs sur la diagonale */ ]);
   opt.minimize(&diagonal_sum);
   ```
3. Objectif : le carré latin 5x5 qui **minimise** la somme des valeurs sur la
   diagonale principale (privilégier les petits nombres sur la diagonale).
4. Vérifiez que le statut reste `SatResult::Sat` et que le coût diagonal obtenu
   est inférieur au reward maximal (178) ci-dessus.


In [12]:
use z3::{Optimize, SatResult};

{
    /// Miroir MINIMISE de la maximisation du carré latin pondéré.
    ///
    /// Rend `None` tant que l'étudiant n'a pas implémenté la résolution :
    /// aucune erreur, aucun panic (règle C.1).
    fn solve_min_diagonal_latin_square_z3(
        n: usize,
        seed: u64,
    ) -> Option<(Vec<Vec<usize>>, i64, SatResult)> {
        // TODO etudiant : copiez la maximisation ci-dessus et adaptez-la
        //   1. Gardez les mêmes contraintes (Optimize, pb_eq one-value-per-cell,
        //      latin rows/cols).
        //   2. Remplacez la maximisation du reward par la minimisation de la
        //      somme des valeurs sur la diagonale principale :
        //        opt.minimize(&diagonal_sum)
        //   3. Rendez (grid, computed_diagonal, status).
        let _ = (n, seed);
        None
    }

    match solve_min_diagonal_latin_square_z3(5, 42) {
        Some((grid, cost, status)) => {
            let libelle = match status {
                SatResult::Sat => "sat",
                SatResult::Unsat => "unsat",
                SatResult::Unknown => "unknown",
            };
            println!("Statut : {} | Cout diagonal minimal : {}", libelle, cost);
            for row in &grid {
                let ligne: Vec<String> = row.iter().map(|v| format!("{:2}", v)).collect();
                println!("{}", ligne.join(" "));
            }
        }
        None => println!("Implementation a completer pour voir le resultat"),
    }
};


Implementation a completer pour voir le resultat


## Synthèse

Ce notebook a présenté deux approches Z3 pour modéliser et résoudre le Sudoku : les entiers symboliques (Int) et les vecteurs de bits (BitVec 4 bits). La comparaison montre que les BitVectors sont plus rapides, car leur domaine restreint (16 valeurs sur 4 bits) permet à Z3 d'utiliser des algorithmes de propagation plus efficaces ; le rapport se lit dans la sortie de la cellule de benchmark, ses valeurs exactes dépendent du matériel. La contrainte `Distinct`, disponible nativement pour les entiers mais nécessitant une décomposition par paires pour les BitVectors, illustre le compromis entre simplicité d'API et performance. Au-delà du Sudoku, Z3 est un outil de résolution SMT polyvalent, applicable à la vérification de programmes, l'analyse de sécurité et la synthèse de code. Les exercices proposés (comptage de solutions, contraintes diagonales, coloration de graphe) montrent comment étendre le modèle de base vers des variantes plus riches.

***

**Navigation** : [<< OR-Tools Python](Sudoku-10-ORTools-Python.ipynb) | [Index](README.md) | [Infer Python >>](Sudoku-15-Infer-Python.ipynb)


## À retenir

**Z3** (prouveur SMT de Microsoft Research, open-source sous licence MIT) résout le Sudoku en **SMT** (Satisfiable Modulo Theories) : on écrit les contraintes en logique du premier ordre et Z3 détermine la satisfaisabilité. Le carnet met en regard deux encodages — **entiers** (`Int`) et **bit-vectors** (`BitVec`).

- Sur le puzzle le plus dur (`hardest[0]`), **Z3 Int** résout la grille — la durée exacte est mesurée à chaque exécution par la cellule de résolution, elle n'est pas figée ici.
- Le contraste d'encodage est net : sur le puzzle 2, le **BitVector** est plusieurs fois plus rapide que l'**Int** — l'écart se lit dans la sortie de la cellule de benchmark, il dépend du matériel.
- Le carnet pousse Z3 au-delà du Sudoku classique : **Sudoku-X** (les deux diagonales **toutes distinctes : True**) et un **carré latin 5x5** optimisé (**reward maximal 178** via `Optimize`).

## Vérifiez votre compréhension

**1. Pourquoi l'encodage BitVec est-il plus rapide que l'encodage Int ?**

<details><summary>Réponse</summary>

Les bit-vectors opèrent sur des représentations binaires compactes et des théories de bits natives, là où les entiers (`Int`) mobilisent une arithmétique plus lourde. C'est une leçon générale d'**encodage** : pour un même problème, la façon de le coder dans le solveur change beaucoup la performance. La cellule de benchmark de ce carnet remesure l'écart à chaque exécution.

</details>

**2. Qu'apporte le Sudoku-X par rapport au Sudoku classique ?**

<details><summary>Réponse</summary>

Deux contraintes de plus : les **deux diagonales principales** doivent aussi avoir 9 valeurs distinctes. La solution trouvée le vérifie — **Diagonale principale tous distincts : True** et **Diagonale secondaire tous distincts : True**. C'est l'illustration que SMT rend *trivial* d'ajouter une contrainte globale : on l'écrit, Z3 la respecte, sans réécrire le solveur.

</details>

**3. Z3 « satisfait » — peut-il aussi « optimiser », comme CP-SAT ?**

<details><summary>Réponse</summary>

Oui, via le contexte `Optimize` : sur le carré latin 5x5, le carnet maximise un reward jusqu'à **178** (miroir de l'exemple OR-Tools de la série). **Statut Z3 Optimize : sat** et **Carré latin valide : True** confirment un optimum atteint et vérifié — Z3 couvre donc aussi le régime *optimisation*, pas seulement la satisfaisabilité.

</details>

## Pour aller plus loin

- **Carnet suivant — Symbolic Automata** : le Sudoku vu comme chaîne déclarative / automate symbolique.

- Le **jumeau C#** du solveur Z3 : `Sudoku-12-Z3-CSharp.ipynb`.

### Lectures

- de Moura & Barrett, *Z3: An Efficient SMT Solver* (2008) — le prouveur SMT utilisé ici.
